# TalkTag Colab quickstart

This notebook installs TalkTag from GitHub and annotates the complete bundled synthetic CHAT transcript. Use a Colab GPU runtime: **Runtime → Change runtime type → T4 GPU**.

In [ ]:
from pathlib import Path
import os
import subprocess

repo = Path("/content/talk-tag")
if repo.exists():
    subprocess.run(["git", "-C", str(repo), "pull", "--ff-only"], check=True)
else:
    subprocess.run(
        ["git", "clone", "https://github.com/OliverHennhoefer/talk-tag.git", str(repo)],
        check=True,
    )
os.chdir(repo)

In [ ]:
%pip install -q -e ".[runtime]"

## Check the runtime

The fixed 4-bit deployment is intended to use CUDA here. The notebook stops early with a clear message if Colab did not attach a GPU.

In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        "No CUDA GPU detected. In Colab, choose Runtime → Change runtime type → T4 GPU, then run all cells again."
    )

print(f"GPU: {torch.cuda.get_device_name(0)}")
subprocess.run(["talk-tag", "doctor", "--device", "cuda"], check=True)

## Inspect the synthetic input

The sample contains no real participant data and deliberately includes child-language forms for TalkTag to review.

In [ ]:
input_path = repo / "examples" / "sample.cha"
print(input_path.read_text(encoding="utf-8"))

## Download the model

The first run downloads several gigabytes of model assets. Loading is deferred to the annotation step so the model is not loaded twice.

In [ ]:
subprocess.run(
    ["talk-tag", "model", "pull", "--device", "cuda", "--no-verify-load"],
    check=True,
)

## Annotate the complete sample

In [ ]:
import shutil

output_dir = repo / "examples" / "sample_out"
shutil.rmtree(output_dir, ignore_errors=True)
subprocess.run(
    [
        "talk-tag",
        "annotate",
        "--input-path",
        str(input_path),
        "--output-dir",
        str(output_dir),
        "--target-speaker",
        "*CHI",
        "--device",
        "cuda",
        "--show-target",
    ],
    check=True,
)

## Review the output

In [ ]:
annotated_path = output_dir / "sample.cha"
print(annotated_path.read_text(encoding="utf-8"))

In [ ]:
import json
from pprint import pprint

report_path = output_dir / "_talk_tag_report.json"
pprint(json.loads(report_path.read_text(encoding="utf-8")))

TalkTag output is intended for assisted annotation and should be reviewed by a human annotator.